# SEBAL Notebook 2: Landsat Data Preparation Workflow
### Study Area: Mississippi Delta
### Acquisition Date: 2020-01-19

This notebook prepares Landsat Collection 2 Level-2 imagery for SEBAL processing over the Mississippi Delta study area.  The workflow includes organizing input scenes, mosaicking overlapping paths/rows, clipping to the AOI boundary, and generating consistent raster outputs for subsequent surface index and energy balance calculations.
1. Import Required Python Libraries and Utility Functions
2. SEBAL Project Directory Structure
3. Configure User Inputs
4. Mosaic and Clip Landsat Bands to MSDelta AOI
5. Process All Landsat Bands

## 1. Import Required Python Libraries and Utility Functions

In [1]:
# Core libraries for raster + vector IO and masking rasters by a polygon boundary
import glob, os
import geopandas as gpd
import rasterio
from datetime import datetime
from Utility import get_file_dataframe
from Utility import get_aoi
from Utility import get_band
from Utility import save_clipped
from Utility import save_mosiac
from Utility import check_info

## 2. Set SEBAL input and output directories

In [2]:
print("Working directory:", os.getcwd())
out_dir = "../02_processed_landsat"
merge_dir = "../02_merge_landsat"
clip_dir = "../03_clip_landsat"

Working directory: G:\Collaborations\Mentee\UF_Anitha Madapakula\Scripts\Python\SEBAL_20200119_MSDelta\00_scripts


## 3. Define bands input files

In [3]:
bands = ["blue","green","red","nir08","swir16","swir22","lwir"]
landsat_file = "../landsat_downloaded_files.csv"
# path to the Mississippi Delta shapefile using a relative path
aoi_shp_file = "../00_shapefiles/MSDelta_UTM15.shp"

## Step 4 — Mosaic and Clip Landsat Bands to MSDelta AOI

In [4]:
band_dict, scenes, df = get_file_dataframe(landsat_file)
print("Selected:", scenes)

                          scene_id      band  \
0  LE07_L2SP_023037_20200119_02_T1      blue   
1  LE07_L2SP_023037_20200119_02_T1     green   
2  LE07_L2SP_023037_20200119_02_T1       red   
3  LE07_L2SP_023037_20200119_02_T1     nir08   
4  LE07_L2SP_023037_20200119_02_T1    swir16   
5  LE07_L2SP_023037_20200119_02_T1    swir22   
6  LE07_L2SP_023037_20200119_02_T1      lwir   
7  LE07_L2SP_023037_20200119_02_T1  qa_pixel   
8  LE07_L2SP_023036_20200119_02_T1      blue   
9  LE07_L2SP_023036_20200119_02_T1     green   

                                                file  
0  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
1  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
2  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
3  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
4  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
5  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
6  G:\Collaborations\Mentee\UF_Anitha Madapakula\...  
7  G:\Collaborations\Mentee\UF_

In [5]:

date_str = scenes[0].split("_")[3]
date_fmt = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")

## Step 5 — Get or define aria of interest

In [6]:
msdelta, msdelta_geom = get_aoi(aoi_shp_file)

MSDelta CRS: EPSG:32615
MSDelta bounds: [ 670003.12528418 3579394.56172946  778512.69077371 3876505.9681242 ]
Geometry ready for raster masking.


## Step 5 — Mosaic and Clip Landsat Bands to MSDelta AOI

In [7]:
date = scenes[0].split("_")[3]   # auto: 20200119

for b in bands:
    print("\nProcessing band:", b)
    
    mosaic_path = save_mosiac(merge_dir, band_dict, scenes, band=b)
    
    src = rasterio.open(mosaic_path)
    save_clipped(
        src,
        aoi_geom=msdelta_geom,
        out_dir=clip_dir,
        band_name=b,
        scene=date,   # or "mosaic_20200119"
        crop=True
    )
    check_info(clip_dir, band_name=b, scene=date)
    src.close()
# NOTE:
# Raster mosaicking can be memory-intensive and unstable after repeated reruns.
# If kernel crashes occur, restart the kernel and rerun this section cleanly.


Processing band: blue
Saved mosaic: ../02_merge_landsat\blue_20200119_mosaic.tif
Original shape: (17681, 10251)
Clipped shape: (9905, 3618)
CRS: EPSG:32615
Deleted: ../03_clip_landsat/blue_20200119_MSDelta.tif
Saved: ../03_clip_landsat/blue_20200119_MSDelta.tif
Opened: ../03_clip_landsat/blue_20200119_MSDelta.tif
CRS: EPSG:32615
Bounds: BoundingBox(left=669975.0, bottom=3579375.0, right=778515.0, top=3876525.0)
Shape: (9905, 3618) Count: 1
Dtype: uint16 Nodata: 0.0

Processing band: green
Saved mosaic: ../02_merge_landsat\green_20200119_mosaic.tif
Original shape: (17681, 10251)
Clipped shape: (9905, 3618)
CRS: EPSG:32615
Deleted: ../03_clip_landsat/green_20200119_MSDelta.tif
Saved: ../03_clip_landsat/green_20200119_MSDelta.tif
Opened: ../03_clip_landsat/green_20200119_MSDelta.tif
CRS: EPSG:32615
Bounds: BoundingBox(left=669975.0, bottom=3579375.0, right=778515.0, top=3876525.0)
Shape: (9905, 3618) Count: 1
Dtype: uint16 Nodata: 0.0

Processing band: red
Saved mosaic: ../02_merge_lands